# Fair ML Pipeline — Demo

End-to-end walkthrough: load data → inject synthetic bias → audit → mitigate → compare results.

**Sensitive attribute:** `Gender`  
**Target:** `Loan_Approval_Status`  
**Dataset:** `../data/loan_dataset.csv`

---
## 0. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Project modules
from analyzer import FairnessAnalyzer
from detection_engine import BiasDetectionEngine
from fair_transformers import InstanceReweighting, CorrelationSuppressor, DisparateImpactRemover
from pipeline_framework import FairPipelineBuilder

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All imports OK.')

---
## 1. Load Dataset

In [ ]:
df_raw = pd.read_csv('../data/loan_dataset.csv')

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
print('Missing values:')
print(df_raw.isnull().sum())
print()
print('Target distribution:')
print(df_raw['Loan_Approval_Status'].value_counts(normalize=True).round(3))
print()
print('Gender distribution:')
print(df_raw['Gender'].value_counts(normalize=True).round(3))

---
## 2. Inject Synthetic Bias

We add calibrated noise to the **Female** subgroup to simulate a biased dataset:

- **Income noise** — Gaussian perturbation that slightly reduces female income on average
- **Credit Score noise** — small downward shift for female applicants
- **Label flip** — a fraction of female approvals are flipped to rejections

This mimics the kind of historical bias often present in real-world lending data.

In [ ]:
df = df_raw.copy()
df.dropna(inplace=True)

female_mask = df['Gender'] == 'Female'
n_female = female_mask.sum()

# --- 2a. Income noise: mean shift of -8% + Gaussian noise
income_noise = np.random.normal(loc=-0.08, scale=0.05, size=n_female)
df.loc[female_mask, 'Income'] = (
    df.loc[female_mask, 'Income'] * (1 + income_noise)
).clip(lower=0)

# --- 2b. Credit Score noise: small downward shift
credit_noise = np.random.normal(loc=-15, scale=8, size=n_female)
df.loc[female_mask, 'Credit_Score'] = (
    df.loc[female_mask, 'Credit_Score'] + credit_noise
).clip(lower=300, upper=850)

# --- 2c. Label flip: 12% of female approvals → rejection
female_approved_idx = df[(female_mask) & (df['Loan_Approval_Status'] == 1)].index
flip_n = int(len(female_approved_idx) * 0.12)
flip_idx = np.random.choice(female_approved_idx, size=flip_n, replace=False)
df.loc[flip_idx, 'Loan_Approval_Status'] = 0

print('Bias injection complete.')
print(f'  Income shift  (Female vs Male): '
      f"{df.loc[female_mask,'Income'].mean():.0f} vs {df.loc[~female_mask,'Income'].mean():.0f}")
print(f'  Credit shift  (Female vs Male): '
      f"{df.loc[female_mask,'Credit_Score'].mean():.1f} vs {df.loc[~female_mask,'Credit_Score'].mean():.1f}")
print(f'  Approval rate (Female vs Male): '
      f"{df.loc[female_mask,'Loan_Approval_Status'].mean():.3f} vs {df.loc[~female_mask,'Loan_Approval_Status'].mean():.3f}")
print(f'  Label flips applied: {flip_n}')

In [ ]:
# Visualise injected bias
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Injected Bias — Female vs Male Distribution', fontsize=13, fontweight='bold')

for gender, color in [('Male', '#4C72B0'), ('Female', '#DD8452')]:
    subset = df[df['Gender'] == gender]
    axes[0].hist(subset['Income'], bins=30, alpha=0.6, label=gender, color=color)
    axes[1].hist(subset['Credit_Score'], bins=30, alpha=0.6, label=gender, color=color)

approval_rates = df.groupby('Gender')['Loan_Approval_Status'].mean()
axes[2].bar(approval_rates.index, approval_rates.values,
            color=['#4C72B0', '#DD8452'], alpha=0.85, edgecolor='white')
axes[2].set_ylim(0, 1)
axes[2].set_ylabel('Approval Rate')

axes[0].set_title('Income Distribution')
axes[1].set_title('Credit Score Distribution')
axes[2].set_title('Approval Rate by Gender')

for ax in axes[:2]:
    ax.legend()

plt.tight_layout()
plt.savefig('bias_injection.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. Bias Audit — FairnessAnalyzer

We use `FairnessAnalyzer` to get a structured audit with confidence intervals before training any model.

In [ ]:
analyzer = FairnessAnalyzer(df=df)
analyzer.set_config(
    target_col='Loan_Approval_Status',
    sensitive_col='Gender',
    positive_label=1
)

# Age binning for intersectional analysis later
analyzer.bin_column(
    column_name='Age',
    bins=[0, 25, 35, 45, 55, 65, float('inf')],
    labels=['0-25', '26-35', '36-45', '46-55', '56-65', '65+'],
    new_column_name='Age_Group'
)

print('FairnessAnalyzer configured.')

In [ ]:
# Selection rate disparity with bootstrap CI
audit = analyzer.get_fairness_audit(n_bootstrap=300)
print('=== Fairness Audit ===')
print(audit)

In [ ]:
# Classification metrics — Fairlearn
fl_results = analyzer.calculate_classification_metrics(engine='fairlearn')
print('=== Fairlearn Metrics ===')
for name, result in fl_results.items():
    ci = result.confidence_interval
    print(f'{name:35s}  value={result.value:+.4f}  '
          f'CI=({ci[0]:+.4f}, {ci[1]:+.4f})  '
          f'effect_size={result.effect_size:.4f}')

In [ ]:
# Intersectional audit: Gender × Age_Group
inter_df = analyzer.intersectional_audit(
    column_names=['Gender', 'Age_Group'],
    apply_fdr_correction=True
)
print('=== Intersectional Audit (Gender × Age_Group) ===')
inter_df.sort_values('p_value_adjusted').head(10)

In [ ]:
# Aequitas disparity table
aequitas_df = analyzer.get_aequitas_metrics()
print('=== Aequitas Disparity Metrics ===')
aequitas_df

In [ ]:
# Visualise audit results
analyzer.generate_report_visualizations(fl_results, output_path='fairness_audit.png')
plt.show()

---
## 4. Detection Engine — Raw Dataset Scan

`BiasDetectionEngine` audits representation, statistical disparity, and proxy correlations before building any pipeline.

In [ ]:
engine = BiasDetectionEngine(
    sensitive_col='Gender',
    target_col='Loan_Approval_Status',
    demographic_benchmarks={'Male': 0.5, 'Female': 0.5},
    proxy_threshold=0.3,
    disparity_threshold=0.1
)

report = engine.run(df)

print('=== Bias Detection Report ===')
print(f'Overall risk : {report.summary["overall_risk"]}')
print(f'Flags raised : {len(report.flags)}')
print()
print('Flags:')
for flag in report.flags:
    print(f'  [{flag["severity"]:6s}] {flag["message"]}')

In [ ]:
print('Representation bias:')
for group, data in report.representation.items():
    print(f'  {group}: observed={data["observed"]:.3f}  benchmark={data["benchmark"]:.3f}  '
          f'gap={data["gap"]:+.3f}')

In [ ]:
print('Top proxy correlations:')
proxy_df = pd.DataFrame(report.proxy_correlations).T
proxy_df.sort_values('correlation', ascending=False).head(8)

---
## 5. Baseline Model — No Mitigation

Train a logistic regression on the biased data to establish a reference point.

In [ ]:
FEATURE_COLS = ['Age', 'Income', 'Credit_Score', 'Loan_Amount', 'Employment_Status']
TARGET_COL   = 'Loan_Approval_Status'
SENSITIVE    = 'Gender'

# Encode categorical columns
df_enc = pd.get_dummies(df[FEATURE_COLS + [TARGET_COL, SENSITIVE]], drop_first=True)

X = df_enc.drop(columns=[TARGET_COL])
y = df_enc[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

print('=== Baseline Model ===')
print(classification_report(y_test, y_pred_baseline))

In [ ]:
# Baseline fairness metrics
gender_col = 'Gender_Male' if 'Gender_Male' in X_test.columns else [c for c in X_test.columns if 'Gender' in c][0]

test_df_with_pred = X_test.copy()
test_df_with_pred['y_true'] = y_test.values
test_df_with_pred['y_pred'] = y_pred_baseline
test_df_with_pred['Gender'] = test_df_with_pred[gender_col].map({1: 'Male', 0: 'Female'})

print('Approval rates by gender (baseline predictions):')
print(test_df_with_pred.groupby('Gender')['y_pred'].mean().round(3))

approval_by_gender = test_df_with_pred.groupby('Gender')['y_pred'].mean()
di_baseline = approval_by_gender.min() / approval_by_gender.max()
print(f'\nDisparate Impact (baseline): {di_baseline:.4f}  (threshold: 0.80)')

---
## 6. Mitigation — Individual Transformers

We test each transformer from `fair_transformers.py` individually.

### 6a. CorrelationSuppressor — Remove Proxy Features

In [ ]:
suppressor = CorrelationSuppressor(
    sensitive_col=SENSITIVE,
    threshold=0.25
)

X_train_num = df_enc.drop(columns=[TARGET_COL]).loc[X_train.index]
X_test_num  = df_enc.drop(columns=[TARGET_COL]).loc[X_test.index]

X_train_sup = suppressor.fit_transform(X_train_num)
X_test_sup  = suppressor.transform(X_test_num)

print(f'Features before suppression : {X_train_num.shape[1]}')
print(f'Features after suppression  : {X_train_sup.shape[1]}')
print(f'Dropped features            : {suppressor.dropped_features_}')

### 6b. DisparateImpactRemover — Quantile Repair

In [ ]:
remover = DisparateImpactRemover(
    sensitive_col=SENSITIVE,
    repair_level=0.8,
    feature_cols=['Income', 'Credit_Score']
)

X_train_rem = remover.fit_transform(
    df[FEATURE_COLS + [SENSITIVE]].loc[X_train.index]
)

# Compare income distribution before vs after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('DisparateImpactRemover — Income Before vs After Repair', fontsize=12)

for gender, color in [('Male', '#4C72B0'), ('Female', '#DD8452')]:
    mask_train = df.loc[X_train.index, 'Gender'] == gender
    axes[0].hist(df.loc[X_train.index][mask_train]['Income'], bins=25,
                 alpha=0.6, label=gender, color=color)
    axes[1].hist(X_train_rem[X_train_rem['Gender'] == gender]['Income'], bins=25,
                 alpha=0.6, label=gender, color=color)

axes[0].set_title('Before')
axes[1].set_title('After (repair_level=0.8)')
for ax in axes:
    ax.legend()
    ax.set_xlabel('Income')

plt.tight_layout()
plt.savefig('disparate_impact_repair.png', dpi=120, bbox_inches='tight')
plt.show()

### 6c. InstanceReweighting — Sample Weights

In [ ]:
reweighter = InstanceReweighting(sensitive_col=SENSITIVE, target_col=TARGET_COL)

train_full = df.loc[X_train.index][FEATURE_COLS + [SENSITIVE, TARGET_COL]]
reweighter.fit(train_full)

print('Sample weight statistics:')
weights = reweighter.sample_weight_
print(f'  mean  : {weights.mean():.4f}  (expected ≈ 1.0)')
print(f'  min   : {weights.min():.4f}')
print(f'  max   : {weights.max():.4f}')
print(f'  sum   : {weights.sum():.1f}  (n = {len(weights)})')

# Train weighted model
mitigated = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
mitigated.fit(X_train, y_train, sample_weight=weights)
y_pred_mitigated = mitigated.predict(X_test)

print('\n=== Mitigated Model (InstanceReweighting) ===')
print(classification_report(y_test, y_pred_mitigated))

---
## 7. Full Pipeline — FairPipelineBuilder

In [ ]:
builder = FairPipelineBuilder.from_config('config.yml')
print(builder.summary())

In [ ]:
result = builder.run(df)

print('=== Pipeline Result ===')
print(f'Bias report risk  : {result.bias_report.summary["overall_risk"]}')
print(f'Pipeline steps    : {[s[0] for s in result.pipeline.steps]}')
print(f'Sample weights    : {"available" if result.sample_weight is not None else "not used"}')

In [ ]:
# Predict on test set using the built pipeline
X_test_raw = df.loc[X_test.index][FEATURE_COLS + [SENSITIVE]]
y_pred_pipeline = result.pipeline.predict(X_test_raw)

print('=== Full Pipeline Predictions ===')
print(classification_report(y_test, y_pred_pipeline))

---
## 8. Comparison — Baseline vs Mitigated

Side-by-side fairness metrics before and after mitigation.

In [ ]:
def approval_rate_by_gender(y_pred, X_test, gender_col):
    df_tmp = X_test[[gender_col]].copy()
    df_tmp['y_pred'] = y_pred
    df_tmp['Gender'] = df_tmp[gender_col].map({1: 'Male', 0: 'Female'})
    return df_tmp.groupby('Gender')['y_pred'].mean()

rates_base = approval_rate_by_gender(y_pred_baseline,  X_test, gender_col)
rates_mit  = approval_rate_by_gender(y_pred_mitigated, X_test, gender_col)

di_base = rates_base.min() / rates_base.max()
di_mit  = rates_mit.min()  / rates_mit.max()

summary = pd.DataFrame({
    'Baseline': rates_base,
    'Mitigated (reweighting)': rates_mit
})
summary.loc['Disparate Impact'] = [di_base, di_mit]
summary.loc['DI passes (≥0.80)'] = [di_base >= 0.80, di_mit >= 0.80]

print('=== Fairness Comparison ===')
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Baseline vs Mitigated — Approval Rate by Gender', fontsize=13, fontweight='bold')

x = np.arange(2)
width = 0.35

for ax, rates, title, di in [
    (axes[0], rates_base, 'Baseline (no mitigation)', di_base),
    (axes[1], rates_mit,  'Mitigated (InstanceReweighting)', di_mit)
]:
    bars = ax.bar(['Female', 'Male'], rates.reindex(['Female', 'Male']).values,
                  color=['#DD8452', '#4C72B0'], alpha=0.85, edgecolor='white', width=0.5)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Approval Rate')
    ax.set_title(title)
    ax.axhline(rates_base.max() * 0.80, color='red', linestyle='--', linewidth=1.2,
               label='80% DI threshold')
    ax.legend(fontsize=9)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
    ax.set_xlabel(f'Disparate Impact = {di:.4f}')

plt.tight_layout()
plt.savefig('comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. CI/CD Gate — assert_fairness

In [ ]:
# Rebuild analyzer on test set predictions for the assert_fairness call
test_result_df = X_test.copy()
test_result_df['Loan_Approval_Status'] = y_test.values
test_result_df['y_pred'] = y_pred_mitigated
test_result_df['Gender'] = test_result_df[gender_col].map({1: 'Male', 0: 'Female'})

analyzer_post = FairnessAnalyzer(df=test_result_df)
analyzer_post.set_config(
    target_col='Loan_Approval_Status',
    sensitive_col='Gender',
    positive_label=1
)

audit_post = analyzer_post.get_fairness_audit(n_bootstrap=200)

try:
    analyzer_post.assert_fairness(audit_post, threshold=0.80, metric='effect_size')
    print('✅ CI/CD gate PASSED — Disparate Impact ≥ 0.80')
except AssertionError as e:
    print(f'❌ CI/CD gate FAILED — {e}')

---
## 10. Summary

| Step | What happened |
|---|---|
| **Bias injection** | Added income/credit noise and label flips to Female subgroup |
| **FairnessAnalyzer** | Measured demographic parity difference, equalized odds, and intersectional disparities with bootstrap CIs |
| **BiasDetectionEngine** | Scanned for representation gaps, statistical disparity, and proxy correlations |
| **CorrelationSuppressor** | Dropped features with high correlation to Gender |
| **DisparateImpactRemover** | Repaired income and credit distributions toward marginal |
| **InstanceReweighting** | Assigned sample weights to balance group/label combinations |
| **FairPipelineBuilder** | Assembled the full pipeline from `config.yml` in 4 lines |
| **assert_fairness** | Verified Disparate Impact ≥ 0.80 as a CI/CD gate |